# 🚀 Entrenamiento LSTM - MEDIUM V3 en Kaggle

**GPU disponible:** Tesla T4 x2 (gratis)
**RAM:** ~30 GB
**Tiempo estimado:** 1-1.5 horas
**Configuración:** 120 días → 7 días

---

## ⚙️ IMPORTANTE: Activar GPU
1. Settings → Accelerator → **GPU T4 x2**
2. Save

---

## 1️⃣ Verificar GPU

In [ ]:
import tensorflow as tf
import gc

gc.collect()

print("="*80)
print("KAGGLE - MEDIUM V3 (120→7 días)")
print("="*80)
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs: {gpus}")

if len(gpus) > 0:
    print(f"\n✅ {len(gpus)} GPU(s) DETECTADA(S)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        tf.config.experimental.set_memory_growth(gpu, True)
    
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f"\n🚀 Multi-GPU: {strategy.num_replicas_in_sync} GPUs")
else:
    print("\n⚠️ NO GPU - Activa en Settings")

print("="*80)

## 2️⃣ Instalar dependencias

In [ ]:
!pip install -q openpyxl seaborn
print("\n✅ Instalado")
import gc
gc.collect()

## 3️⃣ Configurar dataset

**Add Data** → Tu dataset con:
- `train_all_customers_temporal_3.py`
- `online_retail_2.xlsx`

In [ ]:
!ls /kaggle/input/

## 4️⃣ Copiar archivos

In [ ]:
import gc

DATASET_NAME = "lstm-customer-training"  # Ajusta al nombre de tu dataset

!mkdir -p data/processed
!mkdir -p models/temporal/customer_v3/medium

!cp /kaggle/input/{DATASET_NAME}/online_retail_2.xlsx data/processed/
!cp /kaggle/input/{DATASET_NAME}/train_all_customers_temporal_3.py .

print("✅ Archivos copiados")
!ls -lh *.py
gc.collect()

## 5️⃣ Importar script

In [ ]:
import sys
import gc
sys.path.append('.')

from train_all_customers_temporal_3 import CustomerTemporalAnalyzer, TemporalConfig

# Optimización: Batch size reducido
TemporalConfig.MEDIUM['batch_size'] = 32  # Más agresivo que original (64)

print("✅ Importado")
print(f"📊 Config: {TemporalConfig.MEDIUM['window_days']}→{TemporalConfig.MEDIUM['forecast_days']}d, batch {TemporalConfig.MEDIUM['batch_size']}")
gc.collect()

## 6️⃣ Preparar datos

In [ ]:
import warnings
import gc
import numpy as np
from datetime import datetime
warnings.filterwarnings('ignore')

gc.collect()

start_time = datetime.now()
print(f"⏰ Inicio: {start_time}\n")

analyzer = CustomerTemporalAnalyzer(
    data_path='data/processed/online_retail_2.xlsx',
    output_dir='models/temporal/customer_v3'
)

print("="*70)
print("FASE 1: Preparación")
print("="*70)

analyzer.load_and_preprocess_data()
gc.collect()

analyzer.calculate_rfm_metrics()
gc.collect()

analyzer.generate_customer_sequences(min_transactions=5)

# Optimización: Reducir a top 1500 clientes
print(f"\n⚙️ Clientes: {len(analyzer.customers)} → 1500")
analyzer.customers = sorted(analyzer.customers, key=lambda x: x['TotalPurchases'], reverse=True)[:1500]

gc.collect()
print("\n✅ Datos listos")

## 7️⃣ ENTRENAR MEDIUM V3

In [ ]:
import time
import gc

print("="*70)
print("FASE 2: Entrenamiento MEDIUM V3")
print("="*70)

t0 = time.time()

try:
    gc.collect()
    model, history, metrics = analyzer.train_horizon_model(TemporalConfig.MEDIUM)
    
    mins = (time.time() - t0) / 60
    
    print(f"\n✅ COMPLETADO ({mins:.1f} min)")
    print("="*70)
    print(f"Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%")
    print(f"AUC: {metrics['purchase_prob_auc']:.4f}")
    print(f"Days MAE: {metrics['days_mae']:.2f}")
    print(f"Value MAE: ${metrics['value_mae']:.2f}")
    print("="*70)
    
    del model, history
    gc.collect()
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

total = (datetime.now() - start_time).total_seconds() / 60
print(f"\n⏰ Total: {total:.1f} min ({total/60:.1f}h)")

## 8️⃣ Descargar modelos

In [ ]:
!cd models/temporal && zip -r customer_v3_medium_kaggle.zip customer_v3/medium/

print("✅ Comprimido")
print("📥 Descarga desde Output")
!ls -lh models/temporal/*.zip